# Aula 11 — Treinar e Usar um LLM

Três exercícios, um depois de cada microaula. Rode as células na ordem.
Onde aparecer `____`, escreva o que se pede e rode: são experimentos, e
quase nenhum tem resposta única.

**Hoje é o dia do treino.** No fim do Exercício 2 você liga um treino de
35 minutos, que roda durante o intervalo e a terceira microaula. Não
feche a aba enquanto ele roda.

**Travou?** [Colas dos exercícios](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios).

## Preparação

🧰 Baixa as ferramentas do curso e o modelo do professor. Leva uns 20
segundos.

In [ ]:
import os
import shutil
import urllib.request

if os.path.exists("../../data/llm_curso.py"):          # só no PC do professor
    shutil.copy("../../data/llm_curso.py", "llm_curso.py")
else:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/klein-natan/"
                               "nanodegree-AI-Atitus/main/data/llm_curso.py", "llm_curso.py")
from llm_curso import *

professor = modelo_do_professor()
print("Pronto.")

---
## Exercício 1: o treino

📝 Antes: [quiz da Microaula 1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-1-o-treino).
🆘 Travou? [Cola do Exercício 11.1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-1)

🎯 Ver o treino acontecer, e o que o tamanho do passo faz com ele.

### 1.a Um minuto de treino

🧰 `treinar(modelo, minutos)` faz o ciclo da microaula (lote, previsão,
perda, gradiente, passo) com o AdamW, o corte do gradiente e a taxa que
segue o relógio. Escreva o começo de uma frase: a célula mostra o que um
modelo novo escreve com ele, treina por **um minuto** e mostra de novo.
Na primeira vez, o texto de Machado vira tokens (uns 30 segundos a mais).

In [ ]:
comeco = "____"         # o começo de uma frase

modelo = montar_modelo()
print("antes: ", gerar(modelo, comeco, 30), "\n")
historico = treinar(modelo, minutos=1, conferir_a_cada=50)
print("\ndepois:", gerar(modelo, comeco, 30))

In [ ]:
if "____" in comeco:
    print("❌ Escreva o começo de uma frase no lugar do ____.")
elif historico["validacao"][-1] < 6.0:
    print(f"✅ Em um minuto a perda caiu de 6,9 para {historico['validacao'][-1]:.1f}. "
          "O modelo ainda não escreve palavras, mas já acertou espaços, vírgulas e letras comuns.")
else:
    print("🤔 A perda quase não caiu. Rode a célula de novo.")

✏️ O que apareceu primeiro no texto: as palavras, os espaços ou a
pontuação? →

### 1.b O tamanho do passo

A taxa de aprendizado é o tamanho do passo. O padrão é 0.001. Escolha uma
taxa sua, bem maior (como 0.05) ou bem menor (como 0.00001), e compare as
duas, um minuto cada.

In [ ]:
minha_taxa = ____       # um número, como 0.05 ou 0.00001

com_a_minha = treinar(montar_modelo(), minutos=1, taxa=minha_taxa, conferir_a_cada=50)
com_a_padrao = treinar(montar_modelo(), minutos=1, taxa=0.001, conferir_a_cada=50)
desenhar_curva(com_a_minha, com_a_padrao, nomes=[f"taxa {minha_taxa}", "taxa 0.001"])

In [ ]:
minha, padrao = min(com_a_minha["validacao"]), min(com_a_padrao["validacao"])
if minha_taxa > 0.001:
    print(f"Passo grande: {minha:.2f} contra {padrao:.2f}. Passos grandes demais pulam por cima "
          "do lugar bom, e o treino empaca.")
elif minha_taxa < 0.001:
    print(f"Passo pequeno: {minha:.2f} contra {padrao:.2f}. Com passos pequenos demais, o modelo "
          "mal sai do lugar: um minuto não basta.")
else:
    print("Você escolheu a mesma taxa do padrão. Tente 0.05 ou 0.00001.")

---
## Exercício 2: do número à palavra

📝 Antes: [quiz da Microaula 2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-2-do-numero-a-palavra).
🆘 Travou? [Cola do Exercício 11.2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-2)

🎯 Os dois botões de toda ferramenta de IA, no modelo do professor. No
fim, você liga o treino do seu modelo.

### 2.a A temperatura

A célula gera três textos com temperatura 0.2, três com uma temperatura
sua, e três com 1.5. Procure a temperatura em que o texto fica melhor.

In [ ]:
minha_temperatura = ____    # um número entre 0.2 e 1.5

for temperatura in [0.2, minha_temperatura, 1.5]:
    print(f"=== temperatura {temperatura}")
    for _ in range(3):
        print("  ", " ".join(gerar(professor, "A casa de", 30, temperatura=temperatura).split()))

✏️ O que acontece com 0.2? E com 1.5? →

### 2.b O top-k salva a temperatura alta

Com temperatura 1.5, a cauda de tokens improváveis engorda. O top-k deixa
no sorteio só os `k` mais prováveis. Escolha um `k` e compare com o
sorteio sem corte.

In [ ]:
meu_k = ____                # quantos candidatos entram no sorteio, como 10

print("sem corte:", " ".join(gerar(professor, "A casa de", 40, temperatura=1.5).split()))
print(f"\nk = {meu_k}:  ", " ".join(gerar(professor, "A casa de", 40, temperatura=1.5, k=meu_k).split()))

✏️ Com k = 1, o que você espera? Teste. →

### 2.c Ligue o seu treino (antes do intervalo!)

🧰 São 35 minutos. A cada 200 passos aparece uma linha com a perda, e o
melhor modelo fica salvo em `meu_llm.weights.h5`. Para parar antes,
aperte o ■ da célula: o melhor modelo continua salvo.

In [ ]:
keras.utils.set_random_seed(42)
meu_modelo = montar_modelo()
historico = treinar(meu_modelo, minutos=35, arquivo="meu_llm.weights.h5")

---
## Exercício 3: o seu modelo

📝 Antes: [quiz da Microaula 3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-3-ajuste-fino-e-limites).
🆘 Travou? [Cola do Exercício 11.3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-3)

🎯 O seu LLM treinado, comparado com o do professor, e depois ajustado
para escrever sobre outra coisa.

### 3.a O seu modelo contra o do professor

🧰 A curva do seu treino, o melhor modelo carregado do arquivo, e os dois
modelos lado a lado.

In [ ]:
desenhar_curva(historico)
meu_modelo.load_weights("meu_llm.weights.h5")

machado = "Não sei se a senhora sabe que elle não veio hontem."
for nome, m in [("o seu", meu_modelo), ("o do professor", professor)]:
    print(f"=== {nome}: estranheza {estranheza(m, machado):.2f}")
    print(" ".join(gerar(m, "A casa de", 40).split()), "\n")

✏️ O do professor treinou uma hora num processador maior. A diferença
entre os dois é grande? →

### 3.b Ajuste fino: o mesmo treino, outro texto

🧰 `ajuste_fino` faz uma cópia do seu modelo e continua o treino por 200
passos no texto das páginas deste curso (uns 2 minutos). Escreva uma
frase em português de hoje, sobre o que você aprendeu no curso, e veja
quanto cada modelo estranha essa frase.

In [ ]:
minha_frase = "____"        # uma frase sua, sobre o curso

ajustado = ajuste_fino(meu_modelo)
print(f"\nestranheza da sua frase:  original {estranheza(meu_modelo, minha_frase):.2f}   "
      f"ajustado {estranheza(ajustado, minha_frase):.2f}")
print("\nO ajustado continua 'O modelo':", " ".join(gerar(ajustado, "O modelo", 40).split()))

In [ ]:
if "____" in minha_frase:
    print("❌ Escreva uma frase sua no lugar do ____.")
elif estranheza(ajustado, minha_frase) < estranheza(meu_modelo, minha_frase):
    print("✅ A sua frase ficou menos estranha para o modelo ajustado. Nenhuma linha da "
          "arquitetura mudou: só o texto do treino.")
else:
    print("🤔 Para essa frase não mudou. Tente uma com palavras do curso: modelo, dados, treino, perda.")

✏️ O modelo ajustado fala do curso. Ele sabe se o que diz é verdade? →

### 3.c Leve o seu modelo para casa

O Colab apaga tudo quando a aba fecha. Baixe o `meu_llm.weights.h5`
(uns 10 MB): ele carrega em qualquer `montar_modelo()` com
`load_weights`.

In [ ]:
baixar_para_o_computador("meu_llm.weights.h5")

---
## O que você viu hoje

| Experimento | O que ele mostra |
|---|---|
| um minuto de treino | primeiro a forma do texto, depois as palavras |
| a taxa | passo grande demais empaca, pequeno demais não sai do lugar |
| a temperatura | baixa repete, alta inventa palavras |
| o top-k | cortar a cauda salva a temperatura alta |
| o seu modelo | 35 minutos de processador viram um modelo que escreve |
| o ajuste fino | o mesmo treino, outro texto: o modelo é o corpus |